In [101]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [102]:
df = pd.read_csv('../../data/cleaned_df.csv')
df

,ClosePrice,CloseDate,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,Levels_ThreeOrMore,"Levels_ThreeOrMore,MultiSplit","Levels_ThreeOrMore,One",Levels_Two,"Levels_Two,MultiSplit","Levels_Two,MultiSplit,One","Levels_Two,One","Levels_Two,ThreeOrMore","Levels_Two,ThreeOrMore,MultiSplit",Levels_Unknown
0,1800000.0,2025-05-29,1.449186,-0.752775,-0.020604,0.985094,1.223122,1.598201,0.306654,-0.475208,...,False,False,False,True,False,False,False,False,False,False
1,1200000.0,2025-05-19,-0.851357,-0.752775,-0.020789,-1.297509,-0.557954,-1.564054,-0.001721,-0.363324,...,False,False,False,False,False,False,False,False,False,False
2,2250000.0,2025-05-27,-0.878377,-0.752775,-0.020723,-0.754032,-1.448491,-0.509969,-0.310095,1.467957,...,False,False,False,False,False,False,False,False,False,True
3,1425000.0,2025-05-30,-0.359211,-0.752775,-0.020619,0.441617,-0.557954,-0.509969,-0.001721,-0.483690,...,False,False,False,False,False,False,False,False,False,False
4,660000.0,2025-05-30,-0.731698,-0.752775,-0.020797,0.369153,0.332584,-0.509969,-0.001721,-0.890590,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141532,455000.0,2026-05-07,-0.309031,12.922504,-0.020896,-1.949682,-0.557954,-1.564054,-0.618469,-0.539281,...,False,False,False,False,False,False,False,False,False,False
141533,1850000.0,2026-05-08,0.922300,13.773662,0.168023,1.238716,2.113659,-0.509969,-0.001721,-0.202972,...,False,False,False,False,False,False,False,False,False,False
141534,655000.0,2026-05-06,-0.999966,4.410920,-0.020647,-1.043887,-1.448491,-0.509969,-0.001721,-0.261271,...,False,False,False,False,False,False,False,False,False,False
141535,21500000.0,2026-05-15,20.525164,14.965284,0.035679,1.238716,17.252800,6.868626,6.782516,1.776227,...,True,False,False,False,False,False,False,False,False,False


In [104]:
# # Convert CloseDate to datetime
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df = df.sort_values('CloseDate').reset_index(drop=True)


# # Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set


# When modeling, different testing windows will be evaulated
window_options = [3, 6, 9, 12]
for X in window_options:
    train_df, test_df = get_time_split(df, X_months=X)
    
    print(f"--- X = {X} Months ---")
    print(f"Train Date Range: {train_df['CloseDate'].min().strftime('%Y-%m-%d')} to {train_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(train_df)} rows)")
    print(f"Test Date Range:  {test_df['CloseDate'].min().strftime('%Y-%m-%d')} to {test_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(test_df)} rows)\n")
   
    X_train = train_df.drop(columns=['ClosePrice', 'CloseDate'])
    y_train = train_df['ClosePrice']
    X_test = test_df.drop(columns=['ClosePrice', 'CloseDate'])
    y_test = test_df['ClosePrice']

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    print(f'R2 Value: {r2}')
    print(f'MSE Value: {mse}')
    print(f'MAE Value: {mae}')

    print('---------------------------------------')

--- X = 3 Months ---
Train Date Range: 2026-01-30 to 2026-04-29 (31650 rows)
Test Date Range:  2026-04-30 to 2026-05-31 (12434 rows)

R2 Value: 0.2680493416744919
MSE Value: 1495130.5588717894
MAE Value: 630238.5505693287
---------------------------------------
--- X = 6 Months ---
Train Date Range: 2025-10-30 to 2026-04-29 (59678 rows)
Test Date Range:  2026-04-30 to 2026-05-31 (12434 rows)

R2 Value: 0.3315540599748734
MSE Value: 1428799.732201706
MAE Value: 568860.1546419109
---------------------------------------
--- X = 9 Months ---
Train Date Range: 2025-07-30 to 2026-04-29 (94532 rows)
Test Date Range:  2026-04-30 to 2026-05-31 (12434 rows)

R2 Value: 0.35504622151598997
MSE Value: 1403467.9872671051
MAE Value: 544430.999210561
---------------------------------------
--- X = 12 Months ---
Train Date Range: 2025-05-01 to 2026-04-29 (129103 rows)
Test Date Range:  2026-04-30 to 2026-05-31 (12434 rows)

R2 Value: 0.36757282799450075
MSE Value: 1389771.73789041
MAE Value: 543522.151